# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [24]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [25]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    # TODO: implement
    x.requires_grad_(True)
    f = torch.sum(x * x)
    grad_tuple = torch.autograd.grad(outputs=f, inputs=x)
    return grad_tuple[0] 

x = torch.tensor([2.0])
grad_with_autograd_grad(x)
    

tensor([4.])

In [26]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    # TODO: implement
    x.requires_grad_(True)
    x.grad = None
    f = torch.sum(x * x)
    f.backward()
    grad = x.grad.clone()
    x.grad = None
    return grad

x = torch.tensor([2.0])
grad_with_backward(x)


tensor([4.])

In [27]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    # TODO: implement
    a.requires_grad_(True)
    b.requires_grad_(True)
    f = torch.sum(a * a + a * b)
    grad_tuple = torch.autograd.grad(outputs=f, inputs=(a,b))
    return grad_tuple
    
    
a = torch.tensor([3.0])
b = torch.tensor([3.0])
grad_wrt_multiple_inputs(a, b)

(tensor([9.]), tensor([3.]))

## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [28]:
from torch.utils.data import DataLoader, Dataset

In [29]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        # TODO: implement
        self.x = x
        self.y = y

    def __len__(self) -> int:
        # TODO: implement
        return len(self.x)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return self.x[idx], self.y[idx]
    
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([0.0, 1.0])

dataset = TensorPairDataset(x, y)
dataset.__len__()


2

In [30]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        # TODO: implement
        self.input_ids = tokens[:, :-1]
        self.target_ids = tokens[:, 1:]

    def __len__(self) -> int:
        # TODO: implement
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return self.input_ids[idx], self.target_ids[idx]
    
tokens = torch.tensor([[1,2,3,4,5], [10,20,30,40,50]])
dataset = NextTokenDataset(tokens)
dataset[1]

(tensor([10, 20, 30, 40]), tensor([20, 30, 40, 50]))

In [31]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        # TODO: implement
        self.generator = torch.Generator()
        if seed is not None:
            self.generator.manual_seed(seed)
        self.tokens = tokens
        self.crop_len = crop_len

    def __len__(self) -> int:
        # TODO: implement
        return self.tokens.size(0)

    def __getitem__(self, idx: int) -> torch.Tensor:
        # TODO: implement
        start_idx = torch.randint(low=0, high=self.tokens.size(1) - self.crop_len + 1, size=(1,), generator=self.generator)
        end_idx = start_idx + self.crop_len
        return self.tokens[idx, start_idx:end_idx]

tokens = torch.tensor([[1,2,3,4,5], [10,20,30,40,50]])
dataset = RandomCropSequenceDataset(tokens, crop_len=3, seed=1)

for i in range(5):
    print(dataset.__getitem__(0))

tensor([2, 3, 4])
tensor([3, 4, 5])
tensor([1, 2, 3])
tensor([3, 4, 5])
tensor([2, 3, 4])


In [32]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    # TODO: implement
    pass
    lengths = torch.tensor([len(x) for x in seqs])
    T_max = lengths.max()
    diff_to_T_max = [T_max - len(x)for x in seqs]
    padded_tokens = torch.stack([torch.cat([torch.tensor(x, dtype=torch.long), torch.full((diff,), pad_value, dtype=torch.long)]) for x, diff in zip(seqs, diff_to_T_max)])
    padding_mask = torch.arange(T_max) >= lengths.unsqueeze(1) 
    return PaddedBatch(tokens=padded_tokens, lengths=lengths, padding_mask=padding_mask)
    

seqs = list([[1,2,3,4,5], [10,20,30,40], [100,200,300]])
padded_batch = pad_1d_sequences(seqs)
padded_batch


PaddedBatch(tokens=tensor([[  1,   2,   3,   4,   5],
        [ 10,  20,  30,  40,   0],
        [100, 200, 300,   0,   0]]), lengths=tensor([5, 4, 3]), padding_mask=tensor([[False, False, False, False, False],
        [False, False, False, False,  True],
        [False, False, False,  True,  True]]))

In [33]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    # TODO: implement
    lengths = torch.tensor([(len(input_ids), len(target_ids)) for input_ids, target_ids in batch])
    T_max = lengths.max()
    diff_to_T_max = T_max - lengths
    input_ids_padded = torch.stack([torch.cat([x[0], torch.full((diff[0],), pad_value)]) for x, diff in zip(batch, diff_to_T_max)])
    target_ids_padded = torch.stack([torch.cat([x[1], torch.full((diff[1],), pad_value)]) for x, diff in zip(batch, diff_to_T_max)])
    padding_mask = torch.arange(T_max) >= lengths[:,0].unsqueeze(1) 
    attention_mask = ~padding_mask
    return {"input_ids": input_ids_padded, "target_ids": target_ids_padded, "attention_mask": attention_mask, "padding_mask": padding_mask}

    
  

seqs = list([(torch.tensor([1,2,3,4,5]), torch.tensor([10,20,30,40,50])), (torch.tensor([100,200,300]), torch.tensor([1000,2000,3000]))])
collate_next_token_batch(seqs)

    



{'input_ids': tensor([[  1,   2,   3,   4,   5],
         [100, 200, 300,   0,   0]]),
 'target_ids': tensor([[  10,   20,   30,   40,   50],
         [1000, 2000, 3000,    0,    0]]),
 'attention_mask': tensor([[ True,  True,  True,  True,  True],
         [ True,  True,  True, False, False]]),
 'padding_mask': tensor([[False, False, False, False, False],
         [False, False, False,  True,  True]])}

In [34]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    # TODO: implement
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, collate_fn=collate_fn, num_workers=num_workers)

## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [35]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    # TODO: implement
    with torch.no_grad():
      m = torch.zeros_like(p, device=p.device, dtype=p.dtype)
      v = torch.zeros_like(p, device=p.device, dtype=p.dtype)

    t = 0
    return AdamWState(m, v, t);
        

In [36]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    # TODO: implement
    with torch.no_grad():
      state.t += 1
      state.m = betas[0] * state.m + (1 - betas[0]) * grad
      state.v = betas[1] * state.v + (1 - betas[1]) * grad * grad
      m_hat = state.m / (1 - betas[0]**state.t)
      v_hat = state.v / (1 - betas[1]**state.t)
      p -= lr * (m_hat / (torch.sqrt(v_hat) + eps) + weight_decay * p)

    return state




In [37]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    # TODO: implement
    if (torch.isnan(params[0]).any().item()):
      print("NaNs detected")

    return [adamw_step_(p, g, s, lr, betas, eps, weight_decay) for p, g, s in zip(params, grads, states)]

## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [38]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    # TODO: implement
    x, y = batch
    model.train()
    optimizer.zero_grad()
    y_hat = model(x)
    loss = torch.nn.functional.mse_loss(y_hat, y)
    loss.backward()
    optimizer.step()

    return loss.item()



## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [39]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    # TODO: implement
    fan_in = weight.size(1)
    fan_out = weight.size(0)
    return fan_in, fan_out
    

In [40]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    # TODO: implement
    bound = gain * (6 / sum(fan_in_fan_out(weight))) ** 0.5
    return weight.uniform_(-bound, bound)

In [41]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    # TODO: implement
    fan_in, _ = fan_in_fan_out(weight)
    gain = 2.0 ** 0.5
    std = gain / fan_in ** 0.5
    bound = (3.0 ** 0.5) * std
    return weight.uniform_(-bound, bound)
    

In [42]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    # TODO: implement
    if layer.bias is not None:
        nn.init.zeros_(layer.bias)
        
    if scheme == "xavier":
        xavier_uniform_(layer.weight)
    elif scheme == "kaiming_relu":
        kaiming_uniform_(layer.weight, nonlinearity="relu")
    elif scheme == "zero":
        nn.init.zeros_(layer.weight)
    else:
        raise ValueError(f"Unknown scheme: {scheme}")
    return layer